# Every module of the 725 loading projects, imported on the core

`pypi-compat.ipynb` installed the top 1,000 projects with micropip and imported each project's top-level module; 725
of them reached stage `ok`. That probe says whether a package LOADS. It does not say how much of the package works,
because most modules are never imported by the top-level one. This notebook reads the run that imported every module
each of those 725 projects ships (`probe-all.mjs`, driven by `drive.py`): the module list is the project's own
RECORD (`.py` files and `.so` extension modules, without `tests/`, `__main__`, `conftest`), imported one at a time in
a fresh interpreter per project, on the restricted core pinned in `tools/pyodide-dist.json`.

The run was on mlbox, in a docker container whose only route out was an allowlist proxy (pypi.org,
files.pythonhosted.org, cdn.jsdelivr.net), 12 workers, 900 s per project. Nothing below is typed: every figure in the
prose is printed by the cell above it.

Questions: how many projects import completely, what the failures are, which of them are the package's own and which
are an optional dependency that was not installed, what the one crash was, and whether any import reached for the
network.

In [1]:
import json, os, re, sys
from collections import Counter
from pathlib import Path
import pandas as pd
import wmlnb
from IPython.display import Markdown, display

HERE = wmlnb.ROOT / "notebooks" / "python-packages"
DATA = HERE / "data" / "pkg-imports"
DIST = json.loads((wmlnb.ROOT / "tools" / "pyodide-dist.json").read_text())
SUMMARY_SHA256 = "53f343915702b8094a6d87e2236289249beb894febde71594d590f364ae4fa1e"
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 200)

def src_dir():
    d = os.environ.get("PKG_IMPORTS_SRC")
    if not d:
        raise RuntimeError("PKG_IMPORTS_SRC: the folder with pkg-imports-answer.md and pkg-imports/ (slop-zone reports/ui-api)")
    return Path(d)

def read_summary():
    p = src_dir() / "pkg-imports" / "summary.jsonl"
    if wmlnb.sha256_file(p) != SUMMARY_SHA256:
        raise RuntimeError("summary.jsonl is not the file this notebook was written for")
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]

def read_proxy():
    return (src_dir() / "pkg-imports" / "rerun-proxy.log").read_text().splitlines()

def read_events():
    out = []
    for l in (src_dir() / "pkg-imports" / "rerun-huggingface-hub.ts.jsonl").read_text().splitlines():
        t, _, rest = l.partition(" ")
        if not rest.startswith("{"):   # the harness' own "EOF" marker line
            continue
        out.append({"t": t, **json.loads(rest)})
    return out

def read_answer():
    return [(src_dir() / "pkg-imports-answer.md").read_text()]

SRC = {"measured_on": "mlbox, 2026-11-12", "core": DIST["artefact"]["name"], "raw": "slop-zone 27fc603 (rerun: d075ad2)"}
rows = wmlnb.extract(DATA / "summary.json", read_summary, source=SRC | {"file": "pkg-imports/summary.jsonl", "sha256": SUMMARY_SHA256})
proxy = wmlnb.extract(DATA / "rerun-proxy.json", read_proxy, source=SRC | {"file": "pkg-imports/rerun-proxy.log"})
events = wmlnb.extract(DATA / "rerun-events.json", read_events, source=SRC | {"file": "pkg-imports/rerun-huggingface-hub.ts.jsonl"})
answer = wmlnb.extract(DATA / "answer.json", read_answer, source=SRC | {"file": "pkg-imports-answer.md"})[0]
pd.DataFrame(wmlnb.provenance(DATA / "summary.json", DATA / "rerun-proxy.json", DATA / "rerun-events.json", DATA / "answer.json"))

,extract,sha256,rows,extracted_at,commit,core,file,measured_on,raw
0,notebooks/python-packages/data/pkg-imports/summary.json,53f343915702b8094a6d87e2236289249beb894febde71594d590f364ae4fa1e,725,2026-11-12T14:40:58+00:00,aed17fe578fe,pyodide-restricted-314.0.3-35bd1d69,pkg-imports/summary.jsonl,"mlbox, 2026-11-12",slop-zone 27fc603 (rerun: d075ad2)
1,notebooks/python-packages/data/pkg-imports/rerun-proxy.json,617acc2725705ca5,6,2026-11-12T14:40:58+00:00,aed17fe578fe,pyodide-restricted-314.0.3-35bd1d69,pkg-imports/rerun-proxy.log,"mlbox, 2026-11-12",slop-zone 27fc603 (rerun: d075ad2)
2,notebooks/python-packages/data/pkg-imports/rerun-events.json,bcb87fe3b66b8d9e,366,2026-11-12T14:41:12+00:00,aed17fe578fe,pyodide-restricted-314.0.3-35bd1d69,pkg-imports/rerun-huggingface-hub.ts.jsonl,"mlbox, 2026-11-12",slop-zone 27fc603 (rerun: d075ad2)
3,notebooks/python-packages/data/pkg-imports/answer.json,5f54534911053293,1,2026-11-12T14:41:12+00:00,aed17fe578fe,pyodide-restricted-314.0.3-35bd1d69,pkg-imports-answer.md,"mlbox, 2026-11-12",slop-zone 27fc603 (rerun: d075ad2)


## Outcomes

In [2]:
df = pd.DataFrame(rows)
ok = df[df.stage == "ok"]
n_all, n_clean, n_some = len(df), int((ok.failed == 0).sum()), int((ok.failed > 0).sum())
n_crash = int((df.stage == "crash").sum())
n_mod, n_fail = int(ok.modules.sum()), int(ok.failed.sum())
trunc = list(df[df.truncated == True].spec)
display(pd.DataFrame({"projects": {"every module imported": n_clean, "installed, some modules failed": n_some,
                                   "stage crash (no done line)": n_crash, "total": n_all}}))
Markdown(f"Modules attempted: {n_mod}; failed: {n_fail} ({n_fail / n_mod:.1%}). Truncated at the 5,000-module cap: {', '.join(trunc)}. "
         f"Stages: {df.stage.value_counts().astype(int).to_dict()}.")

,projects
every module imported,483
"installed, some modules failed",241
stage crash (no done line),1
total,725


Modules attempted: 53269; failed: 4539 (8.5%). Truncated at the 5,000-module cap: datadog-api-client==2.62.0. Stages: {'ok': 724, 'crash': 1}.

## Failures by error type

In [3]:
fl = pd.DataFrame([{"spec": r["spec"], **f} for r in rows if r.get("failures") for f in r["failures"]])
fl["error"] = fl.error.fillna("")
by_type = fl.groupby("error_type").agg(failed_imports=("m", "size"), projects=("spec", "nunique")).sort_values("failed_imports", ascending=False)
top = by_type.head(10)
other = by_type.iloc[10:]
if len(other):
    top = pd.concat([top, pd.DataFrame({"failed_imports": [other.failed_imports.sum()], "projects": [None]}, index=[f"(other {len(other)} types)"])])
top

,failed_imports,projects
ModuleNotFoundError,3595,211
ImportError,417,56
ImproperlyConfigured,176,3
DidNotEnable,109,1
ZoneInfoNotFoundError,38,2
OSError,28,7
RuntimeError,23,7
AttributeError,21,11
LibraryNotFoundError,20,1
ValueError,15,7


In [4]:
t = ok.set_index("spec")
t.assign(**{"failed / modules": t.failed.astype(int).astype(str) + " / " + t.modules.astype(int).astype(str)}).sort_values("failed", ascending=False).head(10)[["failed / modules"]]

,failed / modules
spec,
mlflow-skinny==3.18.0,422 / 1233
ansible-core==2.22.0,410 / 762
wandb==0.21.0,285 / 725
moto==5.2.5,254 / 1450
mypy==1.19.1,186 / 225
pydantic-ai-slim==2.66.0,160 / 421
django==6.1.3,159 / 875
mlflow-tracing==3.18.0,135 / 603
dask==2026.8.0,130 / 182


In [5]:
df.sort_values("wall_s", ascending=False).head(5)[["spec", "wall_s"]].reset_index(drop=True)

,spec,wall_s
0,mlflow==3.18.0,24.2
1,moto==5.2.5,18.3
2,kubernetes==37.0.2,14.9
3,huggingface-hub==2.4.0,12.0
4,great-expectations==1.24.1,10.6


## What kind of failure: the lane's classification

Each failed import gets one kind from its error alone:

- `ModuleNotFoundError` naming a module of the standard library: **stdlib-missing** (`pwd`, `grp`, `resource`,
  `_multiprocessing`, `tkinter`: the core lacks them, a platform limit);
- `ModuleNotFoundError` naming the module being imported itself: **own-module** (a `.so` or a file the RECORD lists
  but the install does not hold);
- `ModuleNotFoundError` naming anything else: **optional-dep**, a dependency that was not installed (the probe installs
  the project and its REQUIRED closure, not its extras);
- an `ImportError` that says `cannot import name`, `requires` or `install`: **optional-dep-ish**;
- everything else: **other**.

A project whose failures are all optional-dep or optional-dep-ish fails ONLY on optional dependencies.

In [6]:
STD = set(sys.stdlib_module_names)
def kind(f):
    et, err, m = f["error_type"], f["error"] or "", f["m"]
    if et == "ModuleNotFoundError":
        g = re.search(r"No module named '([^']+)'", err)
        name = g.group(1).split(".")[0] if g else "?"
        if name in STD: return "stdlib-missing", name
        if name == m.split(".")[0]: return "own-module", name
        return "optional-dep", name
    if et == "ImportError" and ("cannot import name" in err or "requires" in err.lower() or "install" in err.lower()):
        return "optional-dep-ish", None
    return "other", None
fl[["kind", "missing"]] = fl.apply(lambda f: pd.Series(kind(f)), axis=1)
per_proj = fl.groupby("spec").kind.agg(set)
only_opt = int(per_proj.map(lambda k: k <= {"optional-dep", "optional-dep-ish"}).sum())
n_with = len(per_proj)
display(fl.groupby("kind").agg(failed_imports=("m", "size"), projects=("spec", "nunique")).sort_values("failed_imports", ascending=False))
Markdown(f"{n_with} projects have a failure; {only_opt} fail ONLY on optional dependencies (optional-dep or optional-dep-ish), "
         f"{n_with - only_opt} have a failure of some other kind.")

,failed_imports,projects
kind,,
optional-dep,3017,202
other,600,69
stdlib-missing,530,18
optional-dep-ish,344,47
own-module,48,13


241 projects have a failure; 156 fail ONLY on optional dependencies (optional-dep or optional-dep-ish), 85 have a failure of some other kind.

In [7]:
miss = fl[fl.kind.isin(["optional-dep", "stdlib-missing"])].groupby(["kind", "missing"]).agg(failed_imports=("m", "size"), projects=("spec", "nunique"))
pd.concat([miss.loc[k].sort_values("failed_imports", ascending=False).head(12).assign(kind=k).set_index("kind", append=True) for k in ("optional-dep", "stdlib-missing")])

,,failed_imports,projects
missing,kind,,
numpy,optional-dep,332,12
pygments,optional-dep,208,5
antlr4,optional-dep,207,2
sqlalchemy,optional-dep,173,12
typing_extensions,optional-dep,122,3
flask,optional-dep,111,6
aiohttp,optional-dep,86,13
grpc,optional-dep,84,9
pydantic,optional-dep,75,4


The projects with a failure of another kind, by what the failure says. A project counts under every reason it has.

In [8]:
OTHER_OS = re.compile(r"winreg|_winapi|msvcrt|windll|WindowsError|_overlapped|win32|/System/Library|Security\.framework", re.I)
def reason(f):
    k, name, err = f["kind"], f["missing"], f["error"]
    if k == "stdlib-missing":
        return "another OS (windows, macOS)" if name in ("winreg", "msvcrt", "_winapi", "nt", "_overlapped") else "platform limit: " + name
    if OTHER_OS.search(err) or OTHER_OS.search(f["m"]): return "another OS (windows, macOS)"
    if f["error_type"] == "SystemExit": return "CLI module exits at import (SystemExit)"
    if "processes" in err or "start new thread" in err: return "platform limit: processes / threads"
    if f["error_type"] in ("ImproperlyConfigured", "ZoneInfoNotFoundError", "LibraryNotFoundError", "DidNotEnable"): return "needs configuration, data or a system library: " + f["error_type"]
    if k == "own-module": return "own module missing from the install"
    if k.startswith("optional-dep"): return "(optional dependency)"
    return "other: " + f["error_type"]
fl["reason"] = fl.apply(reason, axis=1)
not_opt = set(per_proj[per_proj.map(lambda k: not k <= {"optional-dep", "optional-dep-ish"})].index)
sub = fl[fl.spec.isin(not_opt) & (fl.reason != "(optional dependency)")]
display(Markdown(f"{len(not_opt)} projects, {len(sub)} failed imports not counting their optional-dependency failures."))
sub.groupby("reason").agg(failed_imports=("m", "size"), projects=("spec", "nunique")).sort_values("projects", ascending=False).head(25)

85 projects, 1197 failed imports not counting their optional-dependency failures.

,failed_imports,projects
reason,,
"another OS (windows, macOS)",52,19
other: ImportError,69,16
own module missing from the install,48,13
other: AttributeError,13,7
other: ValueError,15,7
platform limit: processes / threads,31,5
other: NameError,14,5
other: OSError,4,4
platform limit: tkinter,28,4


## Front-door failures

A module is a front door when its dotted name has at most two parts and none starts with an underscore
(`gunicorn.util`, `joblib.pool`, not `pkg._impl.sub`). These are the imports a user would write. Optional-dep failures
are left out, since a user who has not installed the extra does not expect the module; what remains is a front door
that fails for another reason.

In [9]:
is_front = (fl.m.str.count(r"\.") <= 1) & ~fl.m.str.split(".").map(lambda ps: any(p.startswith("_") for p in ps))
front = fl[is_front & ~fl.kind.str.startswith("optional-dep")]
n_front, n_front_p = len(front), front.spec.nunique()
display(Markdown(f"{n_front} front-door failures that are not an uninstalled optional dependency, in {n_front_p} projects."))
display(front.groupby("reason").agg(failures=("m", "size"), projects=("spec", "nunique")).sort_values("failures", ascending=False).head(15))
front.assign(error=front.error.str.replace("\n", " ", regex=False).str[:90])[["spec", "m", "error_type", "error"]].reset_index(drop=True)

107 front-door failures that are not an uninstalled optional dependency, in 35 projects.

,failures,projects
reason,,
"needs configuration, data or a system library: ImproperlyConfigured",24,2
platform limit: _multiprocessing,17,3
other: RuntimeError,10,1
platform limit: pwd,8,1
other: ImportError,8,7
"another OS (windows, macOS)",7,6
"needs configuration, data or a system library: LibraryNotFoundError",6,1
platform limit: grp,4,2
own module missing from the install,4,2


,spec,m,error_type,error
0,anyio==4.13.0,anyio.to_interpreter,ModuleNotFoundError,No module named '_interpreters'
1,python-dotenv==1.2.4,dotenv.cli,SystemExit,1
2,tqdm==4.67.3,tqdm.tk,ModuleNotFoundError,No module named 'tkinter'
3,pyarrow==22.0.0,pyarrow.cuda,ModuleNotFoundError,No module named 'pyarrow._cuda'
4,pyarrow==22.0.0,pyarrow.libarrow_python,ImportError,dynamic module does not define module export function (PyInit_libarrow_python)
5,lxml==6.0.2,lxml.usedoctest,LookupError,Could not find doctest (only use this function *inside* a doctest)
6,openai==2.30.0,openai.helpers,MissingDependencyError,OpenAI error: missing `numpy` This feature requires additional dependencies:
7,shellingham==1.5.4,shellingham.nt,AttributeError,module 'ctypes' has no attribute 'windll'
8,mcp==2.3.0,mcp.cli,SystemExit,1
9,joblib==1.5.3,joblib.executor,ModuleNotFoundError,No module named '_multiprocessing'


## The crash

One project never reached its `done` line: the Node process exited. `drive.py` records which module had started and not
finished (`died_in`) and the last 400 characters of stderr. For a wasm trap that tail is all stack frames, so the
cause was read from a solo rerun (quoted in the report, pinned here as `answer.json`). `drive.py` now also keeps the
first 400 characters (`stderr_head`); this run's rows predate that field.

In [10]:
crash = df[df.stage == "crash"].iloc[0]
cause = re.search(r"fatal error\. Please report this to the Pyodide maintainers\.\nThe cause of the fatal error was:\n(.+)", answer)
tail = crash.stderr_tail.strip().splitlines()
display(Markdown(f"`{crash.spec}`: died importing `{crash.died_in}` ({crash.wall_s} s into the project). Cause from the solo rerun: `{cause.group(1)}`."))
print("stderr_tail as recorded (its last lines):")
print("\n".join(tail[-4:]))

`pillow==12.2.0`: died importing `PIL._imagingtk` (2.1 s into the project). Cause from the solo rerun: `RuntimeError: function signature mismatch`.

stderr_tail as recorded (its last lines):
  pyodide_fatal_error: true
}

Node.js v26.13.1


## The network attempt

The proxy refused one `CONNECT huggingface.co:443` in the 725-project run, with 12 workers, so it could not be
attributed to a project. The lane asked for `huggingface-hub` alone, one worker, with the proxy log and the probe's
stdout both timestamped to the millisecond. The cells line the refusal up with the probe's events.

In [11]:
deny = [l for l in proxy if "DENY" in l]
t_deny = deny[0].split()[0]
ev = pd.DataFrame(events)
secs = lambda t: pd.Timestamp("2000-01-01 " + t)
ev["dt_ms_from_deny"] = ev.t.map(lambda t: round((secs(t) - secs(t_deny)).total_seconds() * 1000))
near = ev[ev.dt_ms_from_deny.abs() <= 150][["t", "dt_ms_from_deny", "ev", "m", "ok", "s"]]
display(Markdown("Proxy log:\n\n```\n" + "\n".join(proxy) + "\n```"))
display(Markdown(f"{len(deny)} refusal(s): `{deny[0]}`. The probe's events within 150 ms of it:"))
near.reset_index(drop=True)

Proxy log:

```
14:35:58.212 allow: ['cdn.jsdelivr.net', 'files.pythonhosted.org', 'pypi.org']
14:36:08.584 CONNECT cdn.jsdelivr.net:443 ALLOW
14:36:08.939 CONNECT pypi.org:443 ALLOW
14:36:08.981 CONNECT files.pythonhosted.org:443 ALLOW
14:36:10.362 CONNECT huggingface.co:443 DENY
14:36:18.601 CONNECT cdn.jsdelivr.net:443 ALLOW
```

1 refusal(s): `14:36:10.362 CONNECT huggingface.co:443 DENY`. The probe's events within 150 ms of it:

,t,dt_ms_from_deny,ev,m,ok,s
0,14:36:10.297,-65,import,huggingface_hub._tensorboard_logger,False,NaN
1,14:36:10.297,-65,start,huggingface_hub._tree_cache,NaN,NaN
2,14:36:10.297,-65,import,huggingface_hub._tree_cache,True,0.000
3,14:36:10.297,-65,start,huggingface_hub._upload_pipeline,NaN,NaN
4,14:36:10.301,-61,import,huggingface_hub._upload_pipeline,True,0.004
5,14:36:10.301,-61,start,huggingface_hub._webhooks_payload,NaN,NaN
6,14:36:10.303,-59,import,huggingface_hub._webhooks_payload,True,0.002
7,14:36:10.303,-59,start,huggingface_hub._webhooks_server,NaN,NaN
8,14:36:10.304,-58,import,huggingface_hub._webhooks_server,True,0.001
9,14:36:10.304,-58,start,huggingface_hub.cli,NaN,NaN


In [12]:
imp = ev[ev.ev == "import"]
last_start = ev[(ev.ev == "start") & (ev.dt_ms_from_deny <= 0)].iloc[-1]
the_import = imp[imp.m == last_start.m].iloc[0]
allowed = [l for l in proxy if "ALLOW" in l]
Markdown(f"The last import started before the refusal is `{last_start.m}` (started {-last_start.dt_ms_from_deny} ms before it); that import "
         f"returned ok={the_import.ok} after {the_import.s} s, {the_import.dt_ms_from_deny} ms after the refusal: the request was made while the import ran, "
         f"and the import did not wait on it. The rerun's proxy log holds {len(allowed)} allowed CONNECTs and {len(deny)} refusal.")

The last import started before the refusal is `huggingface_hub.cli` (started 58 ms before it); that import returned ok=True after 0.061 s, 3 ms after the refusal: the request was made while the import ran, and the import did not wait on it. The rerun's proxy log holds 4 allowed CONNECTs and 1 refusal.

## Conclusion

In [13]:
top_type = by_type.index[0]
cap = int(df[df.truncated == True].modules.iloc[0])
Markdown("\n".join([
    f"- **{n_clean} of {n_all} projects import every module** ({n_clean / n_all:.1%}); {n_some} have some failure, {n_crash} crashed the interpreter. "
    f"{n_fail} of {n_mod} modules failed ({n_fail / n_mod:.1%}); the most common error is {top_type} ({by_type.loc[top_type, 'failed_imports']} imports in {by_type.loc[top_type, 'projects']} projects).",
    f"- **{only_opt} of the {n_with} failing projects fail only on optional dependencies** that were not installed; the other {n_with - only_opt} have at least one failure that is the core's platform, another OS's module, a CLI module that exits, or a module needing configuration, data or a system library.",
    f"- **{n_front} front-door failures in {n_front_p} projects** are not an uninstalled extra: those are the ones a user would meet.",
    f"- **The crash is `{crash.spec}`**, importing `{crash.died_in}`: `{cause.group(1)}`, a wasm trap. It takes the interpreter down, so every module after it in the list was never tried; the one-module-per-import record needs the process to be restarted after such a trap.",
    f"- **One import reached for the network** and the allowlist refused it: `{last_start.m}`, whose import succeeded without waiting for the request.",
    f"- **Truncated:** {', '.join(trunc)} hit the {cap}-module cap; its modules beyond it were not tried.",
]))

- **483 of 725 projects import every module** (66.6%); 241 have some failure, 1 crashed the interpreter. 4539 of 53269 modules failed (8.5%); the most common error is ModuleNotFoundError (3595 imports in 211 projects).
- **156 of the 241 failing projects fail only on optional dependencies** that were not installed; the other 85 have at least one failure that is the core's platform, another OS's module, a CLI module that exits, or a module needing configuration, data or a system library.
- **107 front-door failures in 35 projects** are not an uninstalled extra: those are the ones a user would meet.
- **The crash is `pillow==12.2.0`**, importing `PIL._imagingtk`: `RuntimeError: function signature mismatch`, a wasm trap. It takes the interpreter down, so every module after it in the list was never tried; the one-module-per-import record needs the process to be restarted after such a trap.
- **One import reached for the network** and the allowlist refused it: `huggingface_hub.cli`, whose import succeeded without waiting for the request.
- **Truncated:** datadog-api-client==2.62.0 hit the 5000-module cap; its modules beyond it were not tried.